# 6. tēma — dzīvokļu sludinājumu datu ieguve no tīmekļa

[![Atvērt Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ValRCS/RTU_BDAA_Course_2026/blob/main/notebooks/lecture_06_web_scraping/06_02_apartment_scraping.ipynb)

Šajā darba burtnīcā izmantosim **SS.com dzīvokļu sludinājumus** kā reālāku web scraping piemēru.

Mērķis nav izveidot industriālu SS.com datu vācēju. Mērķis ir saprast tipisku biznesa datu ieguves procesu:

**vietne → HTTP → HTML tabula → DataFrame → neliela tīrīšana → kopsavilkums → CSV**

Pēc darba burtnīcas jūs pratīsiet:

- lejupielādēt reālas rezultātu lapas HTML;
- pārbaudīt, kādas tabulas lapā ir atrodamas;
- izmantot `pandas.read_html()` strukturētu HTML tabulu nolasīšanai;
- ar BeautifulSoup atrast sludinājumu saites;
- veikt nelielu cenu un platību datu normalizāciju;
- saglabāt iegūtos datus turpmākai analīzei.

> Vietņu HTML struktūra laika gaitā mainās. Ja kāds selektors vairs nedarbojas, tas ir normāls web scraping dzīves cikla piemērs, nevis Python kļūda.


## Darbināšana VS Code un Google Colab

Darba burtnīca ir veidota tā, lai tā darbotos abās vidēs.

### VS Code
- Python;
- VS Code **Python** paplašinājums;
- VS Code **Jupyter** paplašinājums;
- izvēlēts Python kernel.

### Google Colab
Atveriet notebook ar **Open in Colab** pogu augstāk.

Sagatavošanas šūna automātiski instalē tikai tās bibliotēkas, kuru konkrētajā vidē trūkst.


In [1]:
import importlib.util
import subprocess
import sys

required = {
    "requests": "requests",
    "bs4": "beautifulsoup4",
    "pandas": "pandas",
    "lxml": "lxml",
}

missing = [package for module, package in required.items()
           if importlib.util.find_spec(module) is None]

if missing:
    print("Instalējam trūkstošās pakotnes:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
else:
    print("Visas nepieciešamās pakotnes jau ir instalētas.")

print("Python:", sys.version.split()[0])
print("Vide:", "Google Colab" if "google.colab" in sys.modules else "lokāls Jupyter/VS Code")


Visas nepieciešamās pakotnes jau ir instalētas.
Python: 3.14.4
Vide: lokāls Jupyter/VS Code


In [2]:
# i also like to print current time
from datetime import datetime
print("Pašreizējais laiks:", datetime.now().strftime("%Y-%m-%d %H:%M:%S"))

Pašreizējais laiks: 2026-09-14 19:22:04


In [3]:
# standarta bibliotekas
from io import StringIO
from pathlib import Path
from urllib.parse import urljoin

# trešo pušu bibliotekas
import pandas as pd
import requests
from bs4 import BeautifulSoup
from IPython.display import display

# ir labi izdrukāt arī versiju numurus pandam piemēram
print("Pandas versija:", pd.__version__)

pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

HEADERS = {
    "User-Agent": "Mozilla/5.0 (compatible; RTU-BDAA-teaching-example/1.0)"
}

# Varat nomainīt rajonu vai pārdošanu/īri.
URL = "https://www.ss.com/en/real-estate/flats/riga/centre/sell/"

print(URL)


Pandas versija: 3.0.5
https://www.ss.com/en/real-estate/flats/riga/centre/sell/


## 1. Iegūstam HTML ar `requests`

Salīdzinot ar `pd.read_html(URL)`, ir lietderīgi HTML vispirms paņemt pašiem:

- varam norādīt `User-Agent`;
- varam pārbaudīt statusa kodu;
- varam izmantot **to pašu HTML** gan BeautifulSoup, gan Pandas;
- atkļūdošana kļūst saprotamāka.


In [12]:
SAMPLE_HTML = """
<!doctype html>
<html>
<head><title>SS.com apartment sample for classroom fallback</title></head>
<body>
<table>
<thead>
<tr>
<th>Advertisements date</th><th>Street</th><th>R.</th><th>m²</th><th>Floor</th>
<th>Series</th><th>Price, m2</th><th>Price</th>
</tr>
</thead>
<tbody>
<tr><td>Sample listing A</td><td>Kristapa 2</td><td>3</td><td>81</td><td>1/2</td><td>Recon.</td><td>2,963 €</td><td>240,000 €</td></tr>
<tr><td>Sample listing B</td><td>Baldones 28</td><td>2</td><td>40</td><td>4/5</td><td>Lit pr.</td><td>1,722 €</td><td>68,888 €</td></tr>
<tr><td>Sample listing C</td><td>Ranka d. 9</td><td>2</td><td>54</td><td>2/6</td><td>Recon.</td><td>2,315 €</td><td>125,000 €</td></tr>
<tr><td>Sample listing D</td><td>Darba 11a</td><td>1</td><td>29</td><td>1/2</td><td>Stalin project</td><td>1,103 €</td><td>32,000 €</td></tr>
<tr><td>Sample listing E</td><td>M. Nometnu 11</td><td>1</td><td>40</td><td>1/2</td><td>Pre-war house</td><td>925 €</td><td>37,000 €</td></tr>
</tbody>
</table>
</body>
</html>
"""

try:
    # nākamā šūna veic HTTP GET pieprasījumu uz SS.com, lai iegūtu dzīvās lapas HTML saturu.
    response = requests.get(URL, headers=HEADERS, timeout=20)
    print("HTTP statusa kods:", response.status_code)
    response.raise_for_status()
    html = response.text
    source_mode = "live SS.com"

    print("Status:", response.status_code)
    print("Content-Type:", response.headers.get("content-type"))
    print("HTML garums:", len(html))
except requests.RequestException as exc:
    # The notebook remains runnable in both Colab and local Jupyter even if
    # the live site temporarily blocks cloud IPs or is unavailable.
    print("Dzīvo SS.com lapu neizdevās saņemt:", exc)
    print("Turpinām ar nelielu mācību HTML paraugu.")
    html = SAMPLE_HTML
    source_mode = "embedded classroom sample"

print("Datu avots:", source_mode)
print("HTML sākums:")
print(html[:250])


HTTP statusa kods: 200
Status: 200
Content-Type: text/html; charset=UTF-8
HTML garums: 53491
Datu avots: live SS.com
HTML sākums:
<!DOCTYPE html>
<HTML lang="en"><HEAD>
<title>SS.COM Flats - Riga - Centre, Prices, Sell - Advertisements</title>
<meta http-equiv="Content-Type" CONTENT="text/html; charset=UTF-8">
<meta name="viewport" content="user-scalable=1, width=device-wid


In [13]:
print("html mainīgā tips", type(html))

html mainīgā tips <class 'str'>


### Ja vietne bloķē pieprasījumu

Publiska vietne var mainīt aizsardzības vai piekļuves noteikumus. Ja saņemat `403`, `429` vai citu kļūdu:

1. nepalaidiet pieprasījumu ciklā atkārtoti;
2. pārbaudiet URL pārlūkā;
3. apskatiet vietnes noteikumus un `robots.txt`;
4. mācību stundā izmantojiet pasniedzēja saglabātu HTML/CSV paraugu, ja dzīvais avots konkrētajā brīdī nav pieejams.

**429 Too Many Requests** īpaši nozīmē, ka jāsamazina pieprasījumu biežums.


## 2. Apskatām HTML ar BeautifulSoup

Vispirms noskaidrosim, cik `<table>` elementu lapa satur.

SS.com vēsturiski ir izmantojis HTML tabulas arī lapas izkārtojumam, tāpēc lapā var būt vairākas tabulas.


In [14]:
soup = BeautifulSoup(html, "html.parser")

print("Lapas title:", soup.title.get_text(" ", strip=True) if soup.title else "(nav title)")
print("HTML tabulu skaits:", len(soup.find_all("table")))
print("Saišu skaits:", len(soup.find_all("a")))


Lapas title: SS.COM Flats - Riga - Centre, Prices, Sell - Advertisements
HTML tabulu skaits: 6
Saišu skaits: 100


## 3. Strukturētas HTML tabulas ar `pandas.read_html()`

Ja tīmekļa lapā dati jau ir `<table>` struktūrā, nav nepieciešams manuāli lasīt katru `<td>` elementu.

`pd.read_html()`:

- atrod HTML tabulas;
- pārvērš tās DataFrame objektos;
- atgriež **DataFrame sarakstu**.

Svarīgi: tabulas numurs nav stabils API. Tāpēc nepaļaujamies tikai uz `tables[4]`; mēģināsim atrast sludinājumu tabulu pēc kolonnu nosaukumiem.


In [15]:
try:
    tables = pd.read_html(StringIO(html))
except ValueError:
    tables = []

print("Atrastas tabulas:", len(tables))

for i, table in enumerate(tables):
    columns = [str(c) for c in table.columns]
    print(f"{i}: shape={table.shape}, columns={columns[:12]}")


Atrastas tabulas: 6
0: shape=(1, 1), columns=['0']
1: shape=(1, 1), columns=['0']
2: shape=(1, 2), columns=['0', '1']
3: shape=(1, 5), columns=['0', '1', '2', '3', '4']
4: shape=(31, 10), columns=['0', '1', '2', '3', '4', '5', '6', '7', '8', '9']
5: shape=(4, 3), columns=['0', '1', '2']


In [16]:
df = tables[4] # 5. tabula
# izdrukājām izmēru
print("Izmērs:", df.shape)
# pirmās 5 rindiņas
print("Pirmās 5 rindiņas:")
df.head()

Izmērs: (31, 10)
Pirmās 5 rindiņas:


,0,1,2,3,4,5,6,7,8,9
0,Advertisements \tdate,Advertisements \tdate,Advertisements \tdate,Street,R.,m²,Floor,Series,"Price, m2",Price
1,NaN,NaN,Gaišs un plašs dzīvoklis bez remonta renovētā namā. Dzīvokļa p,Lachplesha 35,3,89,3/6,Recon.,"1,656 €","147,378 €"
2,NaN,NaN,"Ļoti gaišs, pilnībā atjaunots 3 istabu dzīvoklis 6. stāvā, foršā",Avotu 8k - 1,3,64,6/6,Pre-war house,"2,469 €","158,000 €"
3,NaN,NaN,Pārdošanā plašs dzīvoklis vienā no pieprasītākajām Rīgas centra,Skolas 22a,3,125,2/6,Pre-war house,"1,816 €","227,000 €"
4,NaN,NaN,Pārdošanā gaišs un mājīgs studijas tipa mansarda dzīvoklis pilnī,Artilerijas 52,1,28,5/5,Recon.,"3,393 €","95,000 €"


In [17]:
# lets move first row as column name
df.columns = df.iloc[0]
# now we can drop the first row
df = df.drop(df.index[0]).reset_index(drop=True)
# show first 5 rows
print("Pirmās 5 rindiņas pēc kolonnu pārvietošanas:")
df.head()

Pirmās 5 rindiņas pēc kolonnu pārvietošanas:


,Advertisements \tdate,Advertisements \tdate,Advertisements \tdate,Street,R.,m²,Floor,Series,"Price, m2",Price
0,NaN,NaN,Gaišs un plašs dzīvoklis bez remonta renovētā namā. Dzīvokļa p,Lachplesha 35,3,89,3/6,Recon.,"1,656 €","147,378 €"
1,NaN,NaN,"Ļoti gaišs, pilnībā atjaunots 3 istabu dzīvoklis 6. stāvā, foršā",Avotu 8k - 1,3,64,6/6,Pre-war house,"2,469 €","158,000 €"
2,NaN,NaN,Pārdošanā plašs dzīvoklis vienā no pieprasītākajām Rīgas centra,Skolas 22a,3,125,2/6,Pre-war house,"1,816 €","227,000 €"
3,NaN,NaN,Pārdošanā gaišs un mājīgs studijas tipa mansarda dzīvoklis pilnī,Artilerijas 52,1,28,5/5,Recon.,"3,393 €","95,000 €"
4,NaN,NaN,Pārdod elegantu klasiskā stila dzīvokli pilnībā renovētā jūgends,Alberta 1,5,161,2/5,Recon.,"4,783 €","770,000 €"


In [18]:
# we can save as csv here with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = Path(f"apartment_listings_{timestamp}.csv")
df.to_csv(output_file, index=False)
print(f"Tabula saglabāta CSV failā: {output_file}")

Tabula saglabāta CSV failā: apartment_listings_20260914_194448.csv


In [19]:
# essentially we only need a few lines of code to accomplis this
# we need URL and we can use pandas to scrape tables
df_list = pd.read_html(URL, header=0)
df = df_list[4] # 5. tabula
# save to csv
df.to_csv("apartment_listings.csv", index=False)


# 3b Atrast saiti uz pēdejo lapu

Mums ir izvilkta pirmā lapa, bet būtu labi izvilkt arī pēdejo un tad varam visas lapas apvienot vienā DataFrame. Lai to izdarītu, mums jāatrod pēdejas lapas numurs.

In [20]:
# how many anchors we have in soup?
anchors = soup.find_all('a')
print("Saišu skaits:", len(anchors))

Saišu skaits: 100


In [21]:
# mūs interesē anchor elements kuram ir atribūta "prev"
prev_anchors = [a for a in anchors if a.get('rel') == 'prev']
print("Iepriekšējās lapas saišu skaits:", len(prev_anchors))

Iepriekšējās lapas saišu skaits: 0


In [22]:
# atradīsm anchor elements kuros ir href kas satur page
page_anchors = [a for a in anchors if a.get('href') and 'page' in a.get('href')]
print("Lapas saišu skaits:", len(page_anchors))

Lapas saišu skaits: 10


In [23]:
# izdrukāsim visus page_anchor elements
print("Lapas saišu elementi:")
for anchor in page_anchors:
    print(anchor)

Lapas saišu elementi:
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page29.html" name="nav_id" rel="prev"><img border="0" height="5" src="https://i.ss.com/img/s_left.png" style="padding-bottom:2px;" width="9"/> Previous</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page2.html" name="nav_id" rel="next">2</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page3.html" name="nav_id" rel="next">3</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page4.html" name="nav_id" rel="next">4</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page5.html" name="nav_id" rel="next">5</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page6.html" name="nav_id" rel="next">6</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page7.html" name="nav_id" rel="next">7</a>
<a class="navi" href="/en/real-estate/flats/riga/centre/sell/page8.html" name="nav_id" rel="next">8</a>
<a class="navi" href="/en/real-es

In [25]:
# find a element in soup which has rel="prev" and get its href attribute
prev_anchor = soup.find('a', rel='prev')
if prev_anchor:
    print("Iepriekšējās lapas href:", prev_anchor.get('href')) 
    href_string = prev_anchor.get('href') 

Iepriekšējās lapas href: /en/real-estate/flats/riga/centre/sell/page29.html


In [26]:
# izmantosim vairākus split lai iegūtu pēdejo lapas numuru
# dalam vispirms ar /
parts = href_string.split('/')
last_part = parts[-1]
print("Pēdējā lapas numurs:", last_part)

Pēdējā lapas numurs: page29.html


In [27]:
# now split by .
parts_dot = last_part.split('.')
first_part_dot = parts_dot[0]
print("Pēdējā lapas numurs (pēc . sadalīšanas):", first_part_dot)


Pēdējā lapas numurs (pēc . sadalīšanas): page29


In [28]:
# now split by page
parts_page = first_part_dot.split('page')
print("Pēdējā lapas numurs (pēc page sadalīšanas):", parts_page[-1])

Pēdējā lapas numurs (pēc page sadalīšanas): 29


In [29]:
# visbeidzot ir jāveic pārveidošana uz int, lai iegūtu pēdējās lapas numuru kā skaitli
last_page_number = int(parts_page[-1])
# now it is an integer!
print("Pēdējā lapas numurs kā skaitlis:", last_page_number, type(last_page_number))

Pēdējā lapas numurs kā skaitlis: 29 <class 'int'>


In [30]:
# tagad kad mums ir pēdejā lapas numurs varam izveidot programmu kas velk ārā visas lapas un saglabā tās csv failos.
# vispirms iegūsim URL visām lapām, izmantojot pēdējā lapas numuru
BASE_URL = "https://www.ss.com/en/real-estate/flats/riga/centre/sell/page"
# we can use list comprehension to generate all page URLs in one row
# all_page_urls = [f"{BASE_URL}{i}.html" for i in range(1, last_page_number + 1)]
# alternatively we can use regular list
all_page_urls = []
for i in range(1, last_page_number + 1):
    page_url = f"{BASE_URL}{i}.html"
    all_page_urls.append(page_url)
print("Visas lapas URL:")
for url in all_page_urls:
    print(url)

Visas lapas URL:
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page1.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page2.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page3.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page4.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page5.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page6.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page7.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page8.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page9.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page10.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page11.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page12.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page13.html
https://www.ss.com/en/real-estate/flats/riga/centre/sell/page14.html
https://www.ss.com/en/real

In [35]:
# now we simply loop through all URLS and read a df from 5th table
# we need a sleep or say 0.5 seconds - so we do not send them all at once
from time import sleep

all_dfs = []
for url in all_page_urls:
    try:
        df_list = pd.read_html(url, header=0) # šeit tiek pieprasīta web lapa un nolasīta tabula
        df = df_list[4] # 5. tabula
        all_dfs.append(df)
        print(f"Tabula no {url} veiksmīgi nolasīta.")
        sleep(0.5)  # Wait 0.5 seconds before the next request
    except Exception as e:
        print(f"Kļūda nolasot tabulu no {url}: {e}")
# cik tabulas ir nolasītas?
print("Nolasīto tabulu skaits:", len(all_dfs))

Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page1.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page2.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page3.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page4.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page5.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page6.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page7.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page8.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page9.html veiksmīgi nolasīta.
Tabula no https://www.ss.com/en/real-estate/flats/riga/centre/sell/page10.html veiksmīgi nolasīta.
Tabula no https://w

In [36]:
# apskatīsm pirmo tabulu
print("Pirma tabula:")
all_dfs[0].head()

Pirma tabula:


,Advertisements \tdate,Advertisements \tdate.1,Advertisements \tdate.2,Street,R.,m²,Floor,Series,"Price, m2",Price
0,NaN,NaN,Продаю 1 комнатную квартиру со всеми удобствами. Зарегистрирован,Bruninieku 92B,1,32,2/3,Pre-war house,"1,125 €","36,000 €"
1,NaN,NaN,2-istabu dzīvoklis ar plašu terasi un skatu uz pievilcīgu pilsēt,Stabu 77,2,79,6/6,Recon.,"1,709 €","135,000 €"
2,NaN,NaN,Gaišs un plašs dzīvoklis bez remonta renovētā namā. Dzīvokļa p,Lachplesha 35,3,89,3/6,Recon.,"1,656 €","147,378 €"
3,NaN,NaN,"Ļoti gaišs, pilnībā atjaunots 3 istabu dzīvoklis 6. stāvā, foršā",Avotu 8k - 1,3,64,6/6,Pre-war house,"2,469 €","158,000 €"
4,NaN,NaN,Pārdošanā plašs dzīvoklis vienā no pieprasītākajām Rīgas centra,Skolas 22a,3,125,2/6,Pre-war house,"1,816 €","227,000 €"


In [37]:
# apskatīsim pēdejo tabulu
print("Pēdējā tabula:")
all_dfs[-1].head()

Pēdējā tabula:


,Advertisements \tdate,Advertisements \tdate.1,Advertisements \tdate.2,Street,R.,m²,Floor,Series,"Price, m2",Price
0,NaN,NaN,"Nomaksa. Investīciju objekts , vienkārši uztaisāms par daudziem",Latgales 108,Other,400,1/3,Priv.house,272 €,"108,900 €"
1,NaN,NaN,Faktiski atsevišķa māja. Pagalma māja. Viss majas otrais stāvs u,Brivibas 234,Other,145,2/2,Spec. pr.,619 €,"89,800 €"
2,NaN,NaN,Īpašnieks pārdod plašu un pilnībā mēbelētu dzīvokli Grostonas ie,Grostonas 21,4,105,7/10,New,"3,762 €","395,000 €"
3,NaN,NaN,Plašas telpas piemērotas dzīvošanai vai birojam. Pārdošanā e,Baznicas 35,6,171,1/6,Pre-war house,"2,104 €","359,750 €"
4,NaN,NaN,"Dzīvoklis, kur Rīgas panorāma kļūst par ikdienas skatu no jūsu l",Brivibas 150,3,85,6/6,Pre-war house,"1,906 €","162,000 €"


In [38]:
# savelkam visas tabulas viena
big_df = pd.concat(all_dfs, ignore_index=True)
print("Visu tabulu apvienotais DataFrame izmērs:", big_df.shape)

Visu tabulu apvienotais DataFrame izmērs: (867, 10)


In [39]:
# nometīsim pirmās divas columns
big_df = big_df.drop(columns=big_df.columns[:2])
print("DataFrame pēc kolonnu dzēšanas izmērs:", big_df.shape)

DataFrame pēc kolonnu dzēšanas izmērs: (867, 8)


In [40]:
# saglabāsim ar laika zīmogu
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = Path(f"apartment_listings_all_{timestamp}.csv")
big_df.to_csv(output_file, index=False)

## 4. Atrodam sludinājumu tabulu

SS.com angļu versijas dzīvokļu tabulā parasti parādās kolonnas, piemēram:

- `Street`
- `R.`
- `m²`
- `Floor`
- `Series`
- `Price, m2`
- `Price`

Izvēlēsimies pirmo tabulu, kurā atrodam gan `Street`, gan kādu `Price` kolonnu.

Šī pieeja ir izturīgāka par konkrēta tabulas indeksa ierakstīšanu kodā.


In [8]:
def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    if isinstance(result.columns, pd.MultiIndex):
        result.columns = [
            " ".join(str(part) for part in col if str(part) != "nan").strip()
            for col in result.columns
        ]
    else:
        result.columns = [str(col).strip() for col in result.columns]

    return result


def find_listing_table(table_list):
    for table in table_list:
        candidate = flatten_columns(table)
        names = [str(c).lower() for c in candidate.columns]

        has_street = any("street" in c for c in names)
        has_price = any("price" in c for c in names)

        if has_street and has_price:
            return candidate
    return None


listing_df = find_listing_table(tables)

if listing_df is None:
    print(
        "Dzīvajā HTML sludinājumu tabulu neatradām. "
        "Iespējams, vietnes struktūra ir mainījusies vai saņemta aizsardzības lapa."
    )
    print("Turpinām ar iebūvēto mācību paraugu, lai pārējās šūnas būtu izpildāmas.")

    html = SAMPLE_HTML
    source_mode = "embedded classroom sample"
    soup = BeautifulSoup(html, "html.parser")
    tables = pd.read_html(StringIO(html))
    listing_df = find_listing_table(tables)

print("Datu avots:", source_mode)
print("Izvēlētās tabulas forma:", listing_df.shape)
display(listing_df.head())


Dzīvajā HTML sludinājumu tabulu neatradām. Iespējams, vietnes struktūra ir mainījusies vai saņemta aizsardzības lapa.
Turpinām ar iebūvēto mācību paraugu, lai pārējās šūnas būtu izpildāmas.
Datu avots: embedded classroom sample
Izvēlētās tabulas forma: (5, 8)


,Advertisements date,Street,R.,m²,Floor,Series,"Price, m2",Price
0,Sample listing A,Kristapa 2,3,81,1/2,Recon.,"2,963 €","240,000 €"
1,Sample listing B,Baldones 28,2,40,4/5,Lit pr.,"1,722 €","68,888 €"
2,Sample listing C,Ranka d. 9,2,54,2/6,Recon.,"2,315 €","125,000 €"
3,Sample listing D,Darba 11a,1,29,1/2,Stalin project,"1,103 €","32,000 €"
4,Sample listing E,M. Nometnu 11,1,40,1/2,Pre-war house,925 €,"37,000 €"


In [9]:
len(tables)

1

## 5. Ātra datu pārbaude

Pirms tīrīšanas vienmēr paskatāmies:

- tabulas izmēru;
- kolonnu nosaukumus;
- pirmās rindas;
- datu tipus;
- trūkstošās vērtības.

Šeit sākas pāreja no **datu ieguves** uz **datu apstrādi**.
Detalizētu tīrīšanu turpināsim nākamajā lekcijā.


In [ ]:
print("Shape:", listing_df.shape)
print()
print("Columns:")
print(listing_df.columns.tolist())
print()
listing_df.info()


## 6. Neliela tīrīšana: cena un platība

No tīmekļa iegūtas skaitliskas vērtības bieži ir teksta formā:

- `240,000 €`
- `2,963 €`
- `81`

Analīzei vēlamies `int` vai `float`.

Izveidosim palīgfunkciju, kas no teksta atstāj tikai ciparus un pēc tam izmanto `pd.to_numeric()`.


In [ ]:
def digits_to_number(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype("string")
        .str.replace(r"[^0-9]", "", regex=True)
        .replace("", pd.NA)
    )
    return pd.to_numeric(cleaned, errors="coerce")


apartments = listing_df.copy()

price_col = next(
    (c for c in apartments.columns if str(c).strip().lower() == "price"),
    None,
)
area_col = next(
    (c for c in apartments.columns if str(c).strip().lower() in {"m²", "m2"}),
    None,
)

if price_col:
    apartments["price_eur"] = digits_to_number(apartments[price_col])

if area_col:
    apartments["area_m2"] = pd.to_numeric(apartments[area_col], errors="coerce")

display(apartments.head())


### Piezīme par cenu formātiem

Šī vienkāršā funkcija ir piemērota **pārdošanas** sludinājumiem, kuros gala cena parasti ir vesels EUR skaitlis.

Īres sludinājumos var parādīties, piemēram, `430 €/mon.`.  
Tur skaitļa iegūšana vēl darbojas, bet papildus būtu jāsaglabā arī **periods** (`month`, `day` u.c.).

Tas ir labs piemērs, kāpēc reālos datos nepietiek tikai ar `replace("€", "")`.


## 7. Vienkāršs kopsavilkums

Vēl neveidosim pilnu datu analīzi — tā būs nākamās tēmas galvenā daļa.

Tomēr jau tagad varam pārbaudīt, vai scraping rezultāts ir jēgpilns.


In [ ]:
if "price_eur" in apartments.columns:
    display(apartments["price_eur"].describe())

if "area_m2" in apartments.columns:
    display(apartments["area_m2"].describe())


In [ ]:
if {"price_eur", "area_m2"}.issubset(apartments.columns):
    apartments["calculated_eur_per_m2"] = (
        apartments["price_eur"] / apartments["area_m2"]
    ).round(2)

    useful_columns = [
        c for c in ["Street", "R.", "area_m2", "price_eur", "calculated_eur_per_m2"]
        if c in apartments.columns
    ]
    display(
        apartments[useful_columns]
        .dropna(subset=["price_eur", "area_m2"])
        .sort_values("calculated_eur_per_m2")
        .head(10)
    )


## 8. Sludinājumu saites ar BeautifulSoup

`read_html()` ir lielisks tabulām, bet tas ne vienmēr saglabā `<a href="...">` saites.

Tāpēc praksē ir normāli kombinēt:

- **Pandas** tabulas struktūrai;
- **BeautifulSoup** HTML atribūtiem un saitēm.

SS.com sludinājumu saites parasti norāda uz URL, kas satur `/msg/`.


In [ ]:
ad_links = []

for a in soup.find_all("a", href=True):
    href = a.get("href")
    if "/msg/" in href:
        full_url = urljoin(URL, href)
        text = a.get_text(" ", strip=True)

        ad_links.append({
            "link_text": text,
            "url": full_url,
        })

links_df = pd.DataFrame(ad_links, columns=["link_text", "url"]).drop_duplicates(subset=["url"])

print("Unikālas sludinājumu saites:", len(links_df))
display(links_df.head(10))


## 9. Kāpēc tabulas rindas un saites nav automātiski jāsavieno pēc indeksa?

Var rasties kārdinājums darīt:

```python
apartments["url"] = links_df["url"]
```

Tas ir droši tikai tad, ja esam pārbaudījuši, ka:

- abu tabulu rindu skaits sakrīt;
- HTML elementu secība tiešām atbilst DataFrame rindām;
- nav papildu reklāmas vai navigācijas saišu.

Profesionālā scraping risinājumā saiti labāk iegūt no **tās pašas sludinājuma rindas**, kuru parsējam, vai izmantot stabilu ID kā savienošanas atslēgu.

Šajā lekcijā svarīgākais ir saprast atšķirību starp **tekstu/tabulu datiem** un **HTML atribūtiem**.


## 10. Saglabājam rezultātu

Saglabājam gan tabulu, gan atrastās saites atsevišķos CSV failos.

Nākamajā lekcijā šo iegūto datu kopu var izmantot tīrīšanai un vizualizācijai.


In [ ]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

apartments_file = OUTPUT_DIR / "ss_apartments_raw_and_lightly_cleaned.csv"
links_file = OUTPUT_DIR / "ss_apartment_links.csv"

apartments.to_csv(apartments_file, index=False, encoding="utf-8-sig")
links_df.to_csv(links_file, index=False, encoding="utf-8-sig")

print("Saglabāts:")
print("-", apartments_file.resolve())
print("-", links_file.resolve())


## 11. Papildu uzdevums — cits rajons

Mainiet tikai URL, piemēram:

```python
https://www.ss.com/en/real-estate/flats/riga/agenskalns/sell/
https://www.ss.com/en/real-estate/flats/riga/purvciems/sell/
https://www.ss.com/en/real-estate/flats/riga/centre/hand_over/
```

Pārbaudiet:

1. vai tabulas struktūra saglabājas;
2. vai mainās kolonnu saturs;
3. kā atšķiras cenas;
4. kas notiek ar cenu parsēšanu īres sludinājumos.


## 12. Papildu uzdevums — vairākas rezultātu lapas

Ja vietnei ir vairākas lapas, scraping kļūst par atkārtojamu procesu:

1. atrod nākamās lapas saiti;
2. pieprasa nākamo lapu;
3. nolasa tabulu;
4. pievieno DataFrame sarakstam;
5. beigās izmanto `pd.concat()`.

Svarīgi:

- neveidojiet ātru bezgalīgu ciklu;
- ievietojiet pauzi starp pieprasījumiem;
- ierobežojiet lapu skaitu mācību eksperimentā;
- beidziet, ja nav nākamās lapas;
- apstrādājiet tīkla kļūdas.

Piemērs idejai:

```python
frames = []

for url in urls:
    # request
    # read_html
    # find listing table
    frames.append(df)
    time.sleep(1)

combined = pd.concat(frames, ignore_index=True)
```

Šo uzdevumu ir vērts pabeigt tikai pēc tam, kad viena lapa darbojas korekti.


## Noslēgums

Šajā lekcijā izmantojām divus savstarpēji papildinošus paņēmienus:

### BeautifulSoup
Labs, ja jāstrādā ar:

- HTML elementiem;
- atribūtiem;
- saitēm;
- nevienmērīgu lapas struktūru.

### `pandas.read_html()`
Ļoti ērts, ja dati jau atrodas HTML `<table>`.

Tipiska darba plūsma:

**requests → HTML → BeautifulSoup / read_html → DataFrame → neliela normalizācija → CSV**

Nākamajā tēmā galvenais jautājums vairs nebūs **“kā dabūt datus?”**, bet gan:

**“kā nekārtīgus iegūtos datus pārveidot uzticamai analīzei un vizualizācijai?”**
